# 00 — Data Preparation: Amazon Books

**RecSys 2026 Tutorial**: From Retrieval to Generation

This notebook downloads the Amazon Books dataset (2023 version via HuggingFace),
applies k-core filtering, creates train/val/test splits, and prepares the shared
data artifacts used by all three paradigm notebooks.

**Dataset**: Amazon Reviews 2023 — Books category (McAuley Lab, UCSD)

**Split strategy**: Leave-last-two-out per user (last = test, second-to-last = val, rest = train)

In [ ]:
!pip install -q datasets pandas numpy pyarrow

In [ ]:
import sys
sys.path.insert(0, '.')

from tutorial_utils import (
    load_amazon_books, create_splits, get_user_histories,
    get_item_titles, get_all_items, DATA_DIR
)
import pandas as pd
import numpy as np

## 1. Download and preprocess

In [ ]:
# This downloads ~2GB on first run. Subsequent runs load from cache.
# Adjust max_users for faster iteration (5000) or full scale.
# Using 20k for preliminary testing
# Will use single seed for tutorial to ensure reproducibility, but feel free to experiment with different seeds.
df = load_amazon_books(max_users=20_000, seed=42)
print(f"Shape: {df.shape}")
print(f"Users: {df['user_idx'].nunique()}")
print(f"Items: {df['item_idx'].nunique()}")
print(f"Interactions: {len(df)}")
df.head()

## 2. Basic EDA

In [ ]:
print("Rating distribution:")
print(df['rating'].value_counts().sort_index())
print(f"\nMean rating: {df['rating'].mean():.2f}")
print(f"Interactions per user: mean={df.groupby('user_idx').size().mean():.1f}, "
      f"median={df.groupby('user_idx').size().median():.1f}")
print(f"Interactions per item: mean={df.groupby('item_idx').size().mean():.1f}, "
      f"median={df.groupby('item_idx').size().median():.1f}")
print(f"\nSparsity: {1 - len(df) / (df['user_idx'].nunique() * df['item_idx'].nunique()):.4%}")

In [ ]:
# Sample titles
item_titles = get_item_titles(df)
sample_items = list(item_titles.items())[:10]
for idx, title in sample_items:
    print(f"  Item {idx}: {title}")

## 3. Create train/val/test splits

In [ ]:
splits = create_splits(df)
user_histories = get_user_histories(splits['train'])

# Ground truth dicts for evaluation
val_ground_truth = {u: it for u, it in splits['val']}
test_ground_truth = {u: it for u, it in splits['test']}

print(f"Users with val data: {len(val_ground_truth)}")
print(f"Users with test data: {len(test_ground_truth)}")
print(f"Users with history: {len(user_histories)}")
print(f"\nExample user 0 history: {[item_titles.get(it, '?') for it in user_histories.get(0, [])[:5]]}")

## 4. Prepare paradigm-specific data formats

Each paradigm notebook expects data in a slightly different format.
We prepare and save all formats here.

In [ ]:
import json
import pickle

# --- Format for GenRec (Paradigm 3): instruction-tuning JSON ---
# for data augmentation for instruction tuning
INSTRUCTION_TEMPLATES = [
    "Based on the books below, what is the next book the user might want to read",
    "Given the user's reading history, predict the next book they will enjoy",
    "Check the books below, recommend a similar book",
    "Using the user's past book selections, what book should they read next",
    "Here is a reading history. Predict the next book the user will choose",
    "Recommend a book based on the following reading list",
    "What book would complement this reading history",
    "Based on these book preferences, suggest the next read",
    "Looking at the books this user has read, predict their next choice",
    "Analyze the reading pattern below and recommend the next book",
]

rng = np.random.RandomState(42)

def make_genrec_data(user_histories, ground_truth, item_titles):
    """Create GenRec-style instruction-tuning JSON."""
    examples = []
    for user_idx, target_item in ground_truth.items():
        history = user_histories.get(user_idx, [])
        if not history:
            continue
        history_titles = [item_titles.get(it, "") for it in history if it in item_titles]
        target_title = item_titles.get(target_item, "")
        if not history_titles or not target_title:
            continue
        examples.append({
            "instruction": rng.choice(INSTRUCTION_TEMPLATES),
            "input": ", ".join(history_titles[-20:]),  # last 20 items max
            "output": target_title,
        })
    return examples

genrec_train = make_genrec_data(user_histories, val_ground_truth, item_titles)
genrec_test = make_genrec_data(
    # For test: include val item in history
    {u: user_histories.get(u, []) + [val_ground_truth[u]]
     for u in test_ground_truth if u in val_ground_truth},
    test_ground_truth,
    item_titles,
)

with open(DATA_DIR / "genrec_train.json", "w") as f:
    json.dump(genrec_train, f, indent=2)
with open(DATA_DIR / "genrec_test.json", "w") as f:
    json.dump(genrec_test, f, indent=2)

print(f"GenRec train examples: {len(genrec_train)}")
print(f"GenRec test examples: {len(genrec_test)}")
print(f"\nSample train example:")
print(json.dumps(genrec_train[0], indent=2)[:500])

In [ ]:
# --- Save shared artifacts ---
shared = {
    "user_histories": user_histories,
    "val_ground_truth": val_ground_truth,
    "test_ground_truth": test_ground_truth,
    "item_titles": item_titles,
    "all_items": list(get_all_items(df)),
    "n_users": df['user_idx'].nunique(),
    "n_items": df['item_idx'].nunique(),
}
with open(DATA_DIR / "shared_data.pkl", "wb") as f:
    pickle.dump(shared, f)

print("All data artifacts saved to", DATA_DIR)
print("Files:", [f.name for f in DATA_DIR.iterdir()])

## 5. Summary

Data artifacts created:
- `data/amazon_books_processed.parquet` — full processed dataset
- `data/splits.pkl` — train/val/test splits
- `data/shared_data.pkl` — user histories, ground truth, item titles
- `data/genrec_train.json` — instruction-tuning format for Paradigm 3
- `data/genrec_test.json` — test set in GenRec format

Next: Run any of the paradigm notebooks (01, 02, 03) — they all load from these shared artifacts.